In [3]:
import openai
import pandas as pd
from qdrant_client import QdrantClient
from qdrant_client.models import Distance, VectorParams, PointStruct

In [4]:
##reading dataset
df_items = pd.read_json('D:/ai-engineering-bootcamp-prerequisites/data/meta_Electronics_2022_2023_with_category_ratings_100_sample_1000.jsonl', orient='records',lines=True)

In [5]:
df_items.head()

,main_category,title,average_rating,rating_number,features,description,price,images,videos,store,categories,details,parent_asin,bought_together,subtitle,author
0,AMAZON FASHION,"Bosttor Bluetooth Beanie Hat with Light, Headl...",4.5,2342,"[100% Acrylic, Elastic closure, Hand Wash Only...",[],26.99,[{'thumb': 'https://m.media-amazon.com/images/...,[{'title': 'Beanie Hat with Bluetooth Headphon...,Bosttor,"[Electronics, Headphones, Earbuds & Accessorie...","{'Department': 'unisex-adult', 'Date First Ava...",B0BH41HYFZ,NaN,NaN,NaN
1,All Electronics,"Aceele USB and USB C to Ethernet Adapter, 3.3f...",4.2,148,[【USB or USB C to Ethernet Adapter】The Etherne...,[],14.99,[{'thumb': 'https://m.media-amazon.com/images/...,[],Aceele,"[Electronics, Computers & Accessories, Network...",{'Product Dimensions': '3.54 x 0.98 x 0.71 inc...,B0BKPB2YQ9,NaN,NaN,NaN
2,All Electronics,HDMI Switch 3 in 1 Out 4K UHD HDMI Switcher Sp...,4.2,3331,[📍3-Port HDMI Switch: This aluminum HDMI switc...,[],18.98,[{'thumb': 'https://m.media-amazon.com/images/...,[{'title': 'Darren reviews VWRHAR 3-1 splitter...,VWRHar,"[Electronics, Home Audio, Home Audio Accessori...",{'Package Dimensions': '6.1 x 4.21 x 0.94 inch...,B09MM5QT3R,NaN,NaN,NaN
3,All Electronics,"Smart Watch,Ip67 Waterproof Bluetooth Smartwat...",3.4,125,[],[],NaN,[{'thumb': 'https://m.media-amazon.com/images/...,[],Burxoe,[],{'Package Dimensions': '3.15 x 3.07 x 2.52 inc...,B09Q5TNDHY,NaN,NaN,NaN
4,Cell Phones & Accessories,"(3 Pack) Cute Airpod Case for Airpods 2&1,3D D...",4.7,335,[【Compatible Airpods 1/2 】This case designed f...,[1],11.99,[{'thumb': 'https://m.media-amazon.com/images/...,"[{'title': 'airpod cases', 'url': 'https://www...",UGUHY,"[Electronics, Headphones, Earbuds & Accessorie...",{'Package Dimensions': '7.44 x 4.61 x 1.57 inc...,B0BZJKX7MS,NaN,NaN,NaN


In [6]:
list(df_items['features'].items())[0]

(0,
 ['100% Acrylic',
  'Elastic closure',
  'Hand Wash Only',
  '【Upgraded Bluetooth Beanie】Bosttor upgraded Bluetooth wireless technology offer much stable and strong connection, support music and calling, easy and fast to pair with your devices. Maximum transmission distance up to 33 feet. Built-in Stereo Speakers & Mic, enhance your music listening experience.',
  '【Long Working Time Music Hat】Built with easy-access USB charging port, simply charge it via the included USD cable. It takes 1-2 hours to get full charged. The battery offers continuous working hours up to 10 hours. Perfect for camping, hiking, skiing, hunting, jogging, cycling, dog walking, auto repair, etc.',
  '【LED Headlight Light Up Your Way】Hands-free LED light is rechargeable and removable. You can remove it to charge by laptop, power bank, socket, car charger, etc. The 4 LED lights can light up to 30 feet away. It points directional light toward what your eyes are looking at. It has 3 brightness settings.',
  '【O

In [7]:
def preprocess_description(row):
    return f"{row['title']} {' '.join(row['features'])}"

In [8]:
def extract_first_large_image(row):
    return row['images'][0].get('large', "")

In [9]:
df_items['preprocessed_description'] = df_items.apply(preprocess_description, axis=1)
df_items['image'] = df_items.apply(extract_first_large_image, axis=1)

In [16]:
df_items['preprocessed_description'][0]

'Bosttor Bluetooth Beanie Hat with Light, Headlamp Cap with Headphones and Built-in Speaker Mic, Gifts for Men Women Teen 100% Acrylic Elastic closure Hand Wash Only 【Upgraded Bluetooth Beanie】Bosttor upgraded Bluetooth wireless technology offer much stable and strong connection, support music and calling, easy and fast to pair with your devices. Maximum transmission distance up to 33 feet. Built-in Stereo Speakers & Mic, enhance your music listening experience. 【Long Working Time Music Hat】Built with easy-access USB charging port, simply charge it via the included USD cable. It takes 1-2 hours to get full charged. The battery offers continuous working hours up to 10 hours. Perfect for camping, hiking, skiing, hunting, jogging, cycling, dog walking, auto repair, etc. 【LED Headlight Light Up Your Way】Hands-free LED light is rechargeable and removable. You can remove it to charge by laptop, power bank, socket, car charger, etc. The 4 LED lights can light up to 30 feet away. It points dir

In [10]:
## sample 50 itesm in the dataset
df_sample = df_items.sample(n=50, random_state=42)

In [18]:
len(df_sample)

50

In [11]:
df_data_to_embed = df_sample[['preprocessed_description', 'image','rating_number','price','average_rating','parent_asin']]

In [13]:
data_to_embed = df_data_to_embed.to_dict(orient = "records")

In [14]:
data_to_embed

[{'preprocessed_description': 'Marame 120mm 5v USB Powered Fan with Speed Controller Cooling for Router Modem Receiver DVR Xbox TV Box (120mm x 120mm x 55mm) 【Solve Your Cooling Problem】\xa0This fan will do the job keeping your devices from overheating and keep your electronics running cool. You can also use them in confined spaces for cooling various electronics. blowing cool air through it to aid in longevity. 【Speed\xa0Control\xa0Switch】 The speed controller located on the cord allows you to adjust the fan’s speed from off to low, medium, and high. This enables you to set the fan to optimal noise and airflow levels for various environments. 【High Compatibility USB-Powered】 Powered by a 3.3ft USB cable. Compatible with desktop, laptop, power bank, AC adapters, car chargers, and other power supplies that support USB connection. USB fan is energy-saving and environmentally friendly. 【Rubber Feet & Dust Filter】\xa0Rubber Feet on the fan raise it up enough off of the surface it is sittin

In [22]:
##define embedding function
def get_embedding(text, model="text-embedding-3-small"):
    response = openai.embeddings.create(
        model=model,
        input=text
    )
    return response.data[0].embedding


In [17]:
## create Qdrant collection
qdrant_client = QdrantClient(url ="http://localhost:6333")

In [28]:
qdrant_client.create_collection(
    collection_name="Amazon-items-collection-01",
    vectors_config=VectorParams(
        size =1536,
        distance=Distance.COSINE)
    
)

True

In [24]:
pointstruct = []
for i,data in enumerate(data_to_embed):
    embedding = get_embedding(data["preprocessed_description"])
    pointstruct.append(
        PointStruct(
        id = i,
        vector = embedding,
        payload = data)
    )

In [25]:
len(pointstruct)

50

In [29]:
##write the embed data to qdrant client
qdrant_client.upsert(
    collection_name="Amazon-items-collection-01",   
    points=pointstruct
)

UpdateResult(operation_id=1, status=<UpdateStatus.COMPLETED: 'completed'>)

In [32]:
##define a function for data retrieval
def retrieve_data(query, k = 5):
    embedding = get_embedding(query)
    results = qdrant_client.query_points(
        collection_name="Amazon-items-collection-01",
        query=embedding,
        limit=k
    )
    return results

In [33]:
retrieve_data("Do you have a USB connectable fan for hot summers")

QueryResponse(points=[ScoredPoint(id=0, version=1, score=0.563146, payload={'preprocessed_description': 'Marame 120mm 5v USB Powered Fan with Speed Controller Cooling for Router Modem Receiver DVR Xbox TV Box (120mm x 120mm x 55mm) 【Solve Your Cooling Problem】\xa0This fan will do the job keeping your devices from overheating and keep your electronics running cool. You can also use them in confined spaces for cooling various electronics. blowing cool air through it to aid in longevity. 【Speed\xa0Control\xa0Switch】 The speed controller located on the cord allows you to adjust the fan’s speed from off to low, medium, and high. This enables you to set the fan to optimal noise and airflow levels for various environments. 【High Compatibility USB-Powered】 Powered by a 3.3ft USB cable. Compatible with desktop, laptop, power bank, AC adapters, car chargers, and other power supplies that support USB connection. USB fan is energy-saving and environmentally friendly. 【Rubber Feet & Dust Filter】\xa